In [94]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)

print("Setup terminé ✓")

Setup terminé ✓


In [ ]:
class BackgammonEnv:
    """Environnement de backgammon complet (15 pions, règles standard).

    Convention : le plateau est toujours représenté du point de vue du
    joueur actif. board[i] > 0 = pions actifs sur le point (i+1),
    board[i] < 0 = pions adverses. Le joueur actif se déplace du point 24
    vers le point 1 puis sort (bear-off). Après un step(), la perspective
    est inversée pour que ce soit toujours au joueur "actif" de jouer.
    """

    N_POINTS = 24
    N_PIONS = 15

    def __init__(self):
        self.board = np.zeros(self.N_POINTS, dtype=int)
        self.bar_actif = 0
        self.bar_adverse = 0
        self.sortis_actif = 0
        self.sortis_adverse = 0
        self.reset()

    def reset(self):
        self.board = np.zeros(self.N_POINTS, dtype=int)

        # Joueur actif : 2 sur le 24, 5 sur le 13, 3 sur le 8, 5 sur le 6
        self.board[23] = 2
        self.board[12] = 5
        self.board[7] = 3
        self.board[5] = 5

        # Adverse (miroir) : 2 sur le 1, 5 sur le 12, 3 sur le 17, 5 sur le 19
        self.board[0] = -2
        self.board[11] = -5
        self.board[16] = -3
        self.board[18] = -5

        self.bar_actif = 0
        self.bar_adverse = 0
        self.sortis_actif = 0
        self.sortis_adverse = 0

        return self._get_state()

    def _get_state(self):
        return {
            "board": self.board.copy(),
            "bar_actif": self.bar_actif,
            "bar_adverse": self.bar_adverse,
            "sortis_actif": self.sortis_actif,
            "sortis_adverse": self.sortis_adverse,
        }

    def tous_pions_dans_maison(self):
        """Le joueur actif a-t-il le droit de sortir des pions (maison = points 1-6) ?"""
        if self.bar_actif > 0:
            return False
        return not np.any(self.board[6:24] > 0)

    def peut_sortir(self, case_depart, de):
        """Règle de bear-off : dé == point exact, ou dé plus grand que le point
        le plus haut occupé de la maison."""
        point = case_depart + 1  # numéro affiché (1-6)
        if point == de:
            return True
        if de > point and not np.any(self.board[point:6] > 0):
            return True
        return False

    def coups_legaux(self, de):
        """Coups légaux du joueur actif pour une seule valeur de dé.
        Un coup est un tuple (depart, arrivee) ; depart == "bar" pour une
        entrée depuis la barre, arrivee == "off" pour une sortie."""
        coups = []

        # Entrée depuis la barre obligatoire avant tout autre coup
        if self.bar_actif > 0:
            case_arrivee = 24 - de
            if self.board[case_arrivee] >= -1:
                coups.append(("bar", case_arrivee))
            return coups

        peut_sortir_maison = self.tous_pions_dans_maison()

        for case_depart in range(24):
            if self.board[case_depart] <= 0:
                continue
            case_arrivee = case_depart - de
            if case_arrivee < 0:
                if peut_sortir_maison and self.peut_sortir(case_depart, de):
                    coups.append((case_depart, "off"))
                continue
            if self.board[case_arrivee] >= -1:
                coups.append((case_depart, case_arrivee))

        return coups

    def appliquer_coup(self, coup):
        """Applique un coup au joueur actif : gère prise et sortie."""
        depart, arrivee = coup

        if depart == "bar":
            self.bar_actif -= 1
        else:
            self.board[depart] -= 1

        if arrivee == "off":
            self.sortis_actif += 1
            return

        if self.board[arrivee] == -1:  # prise d'un pion isolé
            self.board[arrivee] = 0
            self.bar_adverse += 1

        self.board[arrivee] += 1

    def coups_legaux_tour(self, des):
        """Génère toutes les séquences légales de coups pour un tour complet
        (2 dés, ou 4 coups identiques en cas de double). Ne garde que les
        séquences utilisant le nombre maximal de dés jouables, comme l'exige
        la règle officielle."""
        etat_initial = self._sauvegarder_etat()
        meilleures = []
        max_utilises = 0

        def backtrack(des_restants, chemin):
            nonlocal meilleures, max_utilises
            if not des_restants:
                if len(chemin) > max_utilises:
                    max_utilises = len(chemin)
                    meilleures = [chemin.copy()]
                elif len(chemin) == max_utilises and max_utilises > 0:
                    meilleures.append(chemin.copy())
                return

            de = des_restants[0]
            reste = des_restants[1:]
            coups_possibles = self.coups_legaux(de)

            if not coups_possibles:
                backtrack(reste, chemin)
                return

            for coup in coups_possibles:
                sauvegarde = self._sauvegarder_etat()
                self.appliquer_coup(coup)
                backtrack(reste, chemin + [coup])
                self._restaurer_etat(sauvegarde)

        # Avec un double, l'ordre des dés est indifférent (4 coups identiques)
        ordres = [tuple(des)] if des[0] == des[1] else [tuple(des), tuple(des[::-1])]
        for ordre in ordres:
            backtrack(list(ordre), [])

        self._restaurer_etat(etat_initial)

        vues, uniques = set(), []
        for seq in meilleures:
            cle = tuple(seq)
            if cle not in vues:
                vues.add(cle)
                uniques.append(seq)
        return uniques

    def _sauvegarder_etat(self):
        return (self.board.copy(), self.bar_actif, self.bar_adverse,
                self.sortis_actif, self.sortis_adverse)

    def _restaurer_etat(self, etat):
        board, self.bar_actif, self.bar_adverse, self.sortis_actif, self.sortis_adverse = etat
        self.board = board.copy()

    def inverser_perspective(self):
        """Fin de tour : le joueur adverse devient le joueur actif."""
        self.board = -self.board[::-1].copy()
        self.bar_actif, self.bar_adverse = self.bar_adverse, self.bar_actif
        self.sortis_actif, self.sortis_adverse = self.sortis_adverse, self.sortis_actif

    def partie_terminee(self):
        return self.sortis_actif == self.N_PIONS or self.sortis_adverse == self.N_PIONS

    def step(self, coups):
        """Joue une séquence de coups pour le joueur actif puis change de perspective."""
        for coup in coups:
            self.appliquer_coup(coup)

        gagnant = None
        if self.sortis_actif == self.N_PIONS:
            gagnant = "actif"
        elif self.sortis_adverse == self.N_PIONS:
            gagnant = "adverse"

        self.inverser_perspective()
        return self._get_state(), gagnant

In [96]:
def obtenir_coordonnees_case(i):
    """
    i = indice interne (0-23), point affiche = i+1 (convention standard 1-24).
    Layout : bas-droite (1-6), bas-gauche (7-12), haut-gauche (13-18), haut-droite (19-24)
    """
    if i <= 5:              # points 1-6, bas-droite
        x = 12 - i
        y_base, direction = 0, 1
    elif i <= 11:            # points 7-12, bas-gauche
        x = 11 - i
        y_base, direction = 0, 1
    elif i <= 17:            # points 13-18, haut-gauche
        x = i - 12
        y_base, direction = 7, -1
    else:                    # points 19-24, haut-droite
        x = i - 11
        y_base, direction = 7, -1
    return x + 0.5, y_base + direction * 1.5

In [ ]:
def afficher_plateau(board, bar_actif=0, bar_adverse=0, sortis_actif=0, sortis_adverse=0):
    fig, ax = plt.subplots(figsize=(13, 7))

    grand_cadre = patches.Rectangle((-0.7, -0.7), 15.4, 8.4, facecolor="none", edgecolor="black", linewidth=2)
    ax.add_patch(grand_cadre)

    for coords in [(0, 4), (7, 4), (7, 0), (0, 0)]:
        rect = patches.Rectangle(coords, 6, 3, facecolor="none", edgecolor="black", linewidth=1)
        ax.add_patch(rect)

    ax.plot([6.5, 6.5], [-0.3, 7.3], color="black", linewidth=1.5)

    for i in range(24):
        if i <= 5:
            x = 12 - i; y_base = 0; direction = 1
        elif i <= 11:
            x = 11 - i; y_base = 0; direction = 1
        elif i <= 17:
            x = i - 12; y_base = 7; direction = -1
        else:
            x = i - 11; y_base = 7; direction = -1

        triangle = patches.Polygon(
            [(x, y_base), (x + 1, y_base), (x + 0.5, y_base + direction * 3)],
            closed=True, facecolor="none", edgecolor="black", linewidth=0.8
        )
        ax.add_patch(triangle)

        y_num = y_base + direction * 0.2
        ax.text(x + 0.5, y_num, str(i + 1), fontsize=7, ha='center', color="gray")  # i+1 = numero affiche

        n_pions = int(board[i])
        couleur_pion = "white" if n_pions > 0 else "black"
        n_abs = abs(n_pions)
        espace = 0.55 if n_abs <= 5 else 2.7 / n_abs
        for p in range(n_abs):
            y_pion = y_base + direction * (0.55 + p * espace)
            cercle = patches.Circle((x + 0.5, y_pion), 0.22, facecolor=couleur_pion, edgecolor="black", linewidth=0.8)
            ax.add_patch(cercle)

    # Pions sur la barre (au centre du plateau)
    for p in range(bar_actif):
        cercle = patches.Circle((6.5, 3.2 + p * 0.5), 0.22, facecolor="white", edgecolor="black", linewidth=0.8)
        ax.add_patch(cercle)
    for p in range(bar_adverse):
        cercle = patches.Circle((6.5, 3.8 - p * 0.5), 0.22, facecolor="black", edgecolor="black", linewidth=0.8)
        ax.add_patch(cercle)

    ax.set_xlim(-1, 15)
    ax.set_ylim(-1.2, 8.2)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(
        f"Bar actif: {bar_actif}  |  Bar adverse: {bar_adverse}  |  "
        f"Sortis actif: {sortis_actif}  |  Sortis adverse: {sortis_adverse}",
        fontsize=11
    )
    plt.tight_layout()
    return fig, ax

In [ ]:
def verifier_invariant(env):
    n_actif = np.sum(env.board[env.board > 0]) + env.bar_actif + env.sortis_actif
    n_adverse = np.sum(np.abs(env.board[env.board < 0])) + env.bar_adverse + env.sortis_adverse
    assert n_actif == env.N_PIONS, f"Erreur : {n_actif} pions actifs au lieu de {env.N_PIONS}"
    assert n_adverse == env.N_PIONS, f"Erreur : {n_adverse} pions adverses au lieu de {env.N_PIONS}"
    print(f"Invariant respecté : {n_actif} pions actifs, {n_adverse} pions adverses ✓")

In [99]:
def lancer_des():
    de1 = np.random.randint(1, 7)
    de2 = np.random.randint(1, 7)
    if de1 == de2:
        return [de1, de1, de1, de1]
    return [de1, de2]

In [ ]:
env = BackgammonEnv()
state = env.reset()
verifier_invariant(env)

des = lancer_des()
print("Dés :", des)

sequences = env.coups_legaux_tour(des)
print(f"{len(sequences)} séquence(s) de coups optimale(s) trouvée(s)")
if sequences:
    print("Exemple de séquence jouée :", sequences[0])
    state, gagnant = env.step(sequences[0])
    if gagnant:
        print(f"Partie terminée, gagnant : {gagnant}")
else:
    print("Aucun coup jouable avec ce dé, passage du tour.")
    env.inverser_perspective()
    state = env._get_state()

afficher_plateau(
    state["board"], state["bar_actif"], state["bar_adverse"],
    state["sortis_actif"], state["sortis_adverse"]
)
plt.show()